In [1]:
# RSA on text message

import math
import random
from typing import Tuple

def generate_prime(min_value: int, max_value: int) -> int:
    """Generate a prime number between min_value and max_value."""
    def is_prime(n: int) -> bool:
        if n < 2:
            return False
        for i in range(2, int(math.sqrt(n)) + 1):
            if n % i == 0:
                return False
        return True

    prime = random.randrange(min_value, max_value)
    while not is_prime(prime):
        prime = random.randrange(min_value, max_value)
    return prime

def generate_keypair(p: int, q: int) -> Tuple[Tuple[int, int], Tuple[int, int]]:
    """Generate public and private keypairs."""
    n = p * q
    phi = (p - 1) * (q - 1)

    # Choose e: coprime to phi and 1 < e < phi
    e = random.randrange(1, phi)
    while math.gcd(e, phi) != 1:
        e = random.randrange(1, phi)

    # Calculate d: modular multiplicative inverse of e
    def extended_gcd(a: int, b: int) -> Tuple[int, int, int]:
        if a == 0:
            return b, 0, 1
        gcd, x1, y1 = extended_gcd(b % a, a)
        x = y1 - (b // a) * x1
        y = x1
        return gcd, x, y

    _, d, _ = extended_gcd(e, phi)
    d = d % phi
    if d < 0:
        d += phi

    return ((e, n), (d, n))

def encrypt(public_key: Tuple[int, int], plaintext: str) -> list:
    """Encrypt the plaintext using public key."""
    e, n = public_key
    # Convert each character to number and encrypt
    cipher = [(ord(char) ** e) % n for char in plaintext]
    return cipher

def decrypt(private_key: Tuple[int, int], ciphertext: list) -> str:
    """Decrypt the ciphertext using private key."""
    d, n = private_key
    # Decrypt each number and convert back to character
    plain = [chr((char ** d) % n) for char in ciphertext]
    return ''.join(plain)

def main():
    # Generate two prime numbers
    p = generate_prime(100, 1000)
    q = generate_prime(100, 1000)

    print(f"Generated prime numbers: p = {p}, q = {q}")

    # Generate public and private keys
    public_key, private_key = generate_keypair(p, q)
    print(f"Public key: {public_key}")
    print(f"Private key: {private_key}")

    # Get message from user
    message = input("Enter a message to encrypt: ")

    # Encrypt the message
    encrypted_msg = encrypt(public_key, message)
    print(f"Encrypted message: {encrypted_msg}")

    # Decrypt the message
    decrypted_msg = decrypt(private_key, encrypted_msg)
    print(f"Decrypted message: {decrypted_msg}")

if __name__ == "__main__":
    main()

Generated prime numbers: p = 227, q = 151
Public key: (19631, 34277)
Private key: (4571, 34277)
Enter a message to encrypt: hello1234
Encrypted message: [1864, 6464, 24592, 24592, 19771, 12470, 12411, 24262, 6721]
Decrypted message: hello1234


In [ ]:
import random
from math import gcd
from PIL import Image
import numpy as np

def generate_keypair(p, q):
    n = p * q
    phi = (p - 1) * (q - 1)
    e = random.randrange(1, phi)
    g = gcd(e, phi)
    while g != 1:
        e = random.randrange(1, phi)
        g = gcd(e, phi)
    d = pow(e, -1, phi)
    return ((e, n), (d, n))

def encrypt(pk, plaintext):
    key, n = pk
    cipher = [pow(int(byte), key, n) for byte in plaintext]
    return cipher

def decrypt(pk, ciphertext):
    key, n = pk
    plain = [pow(char, key, n) for char in ciphertext]
    return plain

def encrypt_image(image_path, public_key):
    with Image.open(image_path) as img:
        img = img.convert('L')
        data = np.array(img)
    shape = data.shape
    flattened = data.flatten()
    print(f"Original flattened size: {len(flattened)}")
    encrypted = encrypt(public_key, flattened)
    print(f"Encrypted size: {len(encrypted)}")
    return encrypted, shape

def decrypt_image(encrypted, private_key, shape):
    decrypted = decrypt(private_key, encrypted)
    print(f"Decrypted size: {len(decrypted)}")
    if len(decrypted) != np.prod(shape):
        raise ValueError("Decrypted data size mismatch!")
    reconstructed = np.array(decrypted).reshape(shape)
    return Image.fromarray(reconstructed.astype('uint8'))
def visualize_encrypted(encrypted, shape):
    encrypted_normalized = np.array(encrypted)
    encrypted_normalized = (encrypted_normalized - encrypted_normalized.min()) * (255 / (encrypted_normalized.max() - encrypted_normalized.min()))
    return Image.fromarray(encrypted_normalized.reshape(shape).astype('uint8'))

p = 61
q = 53
public, private = generate_keypair(p, q)

image_path = (r"/content/building.jpg")
encrypted, shape = encrypt_image(image_path, public)
visualized_image = visualize_encrypted(encrypted, shape)
visualized_image.save("rsaimg_encrypted_visualization.png")
decrypted_image = decrypt_image(encrypted, private, shape)
decrypted_image.save("rsaimg_decrypted.png")
original_image = Image.open(image_path)
original_image.save("original_rsaimage.png")
print("Encryption and decryption complete. Check the output images.")
